<a href="https://colab.research.google.com/github/d-urbonas/CMU-MLinProd/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You are evaluating a candidate sentiment model to replace a production baseline. Your goal is to determine whether this model should ship.

“Ship” means: we would choose the candidate model over the baseline for deployment based on the evidence you collect.

### Step 1 — Prepare the tested Colab environment

Use a fresh Colab CPU runtime with Python 3.12.
Run the installation cell once, then run the version check before continuing.

In [1]:
import subprocess
import sys

PINNED_PACKAGES = [
    "datasets==4.8.5",
    "emoji==2.15.0",
    "pandas==2.3.3",
    "pyarrow==25.0.1",
    "scikit-learn==1.7.2",
    "torch==2.9.1+cpu",
    "tqdm==4.67.1",
    "transformers==4.57.6",
    "wandb==0.28.2",
]
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    *PINNED_PACKAGES,
])

0

In [2]:
import wandb

# Paste your W&B key only into the hidden prompt opened by this command.
# Never type a token into a code cell or save it in the notebook.
wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: durb (durb-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

The login prompt stores the key only in the current runtime.
If you accidentally typed a key into a cell, revoke it in W&B before continuing.

In [3]:
import platform
import sys

import datasets
import emoji
import pandas as pd
import pyarrow
import sklearn
import torch
import tqdm
import transformers
import wandb

versions = {
    "python": platform.python_version(),
    "datasets": datasets.__version__,
    "emoji": emoji.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "scikit-learn": sklearn.__version__,
    "torch": torch.__version__,
    "tqdm": tqdm.__version__,
    "transformers": transformers.__version__,
    "wandb": wandb.__version__,
}
assert (3, 12) <= sys.version_info[:2] < (3, 14), "Use the tested Python 3.12 runtime."
pd.Series(versions, name="version")

,version
python,3.12.13
datasets,4.8.5
emoji,2.15.0
pandas,2.3.3
pyarrow,25.0.1
scikit-learn,1.7.2
torch,2.9.1+cpu
tqdm,4.67.1
transformers,4.57.6
wandb,0.28.2


In [ ]:
# !pip uninstall -y torchvision
# !pip install -q -U transformers datasets emoji
#
# Then select Runtime → Restart session in Google Colab. Restarting is required; simply rerunning the import cell may retain the broken module state.

In [4]:
import os
import re

import emoji
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
np.random.seed(SEED)

PROJECT = "mlip-lab4-slices-2026"
ENTITY = None
RUN_NAME = "baseline_vs_candidate"


In [5]:
# Models to compare
MODELS = {
    "baseline_model": "cardiffnlp/twitter-roberta-base-sentiment-latest",
    "candidate_model": "LYTinn/finetuning-sentiment-model-tweet-gpt2",
}

In [6]:
# Label normalization for tweet_eval (0/1/2 -> string labels)
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

# Many HF sentiment models output labels like LABEL_0 / LABEL_1 / LABEL_2
HF_LABEL_MAP = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

USE_HF_DATASET = True  # set False to use tweets.csv fallback

### Step 2 - Load a dataset from Hugging Face

In [7]:
if USE_HF_DATASET:
    ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")
    df = pd.DataFrame(ds["test"]).head(500).copy()
    df["label"] = df["label"].map(ID2LABEL)
else:
    fallback_url = "https://raw.githubusercontent.com/mlip-cmu-online/lab-model-testing/main/tweets.csv"
    df = pd.read_csv(fallback_url)
    # Ensure it has 'text' and 'label' columns
    df = df.rename(columns={c: c.strip() for c in df.columns})
    assert {"text","label"}.issubset(df.columns), "tweets.csv must include text,label"
    df["label"] = df["label"].astype(str).str.lower()

df = df[["text","label"]].dropna().reset_index(drop=True)
df.head(3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

,text,label
0,@user @user what do these '1/2 naked pics' hav...,neutral
1,OH: “I had a blue penis while I was this” [pla...,neutral
2,"@user @user That's coming, but I think the vic...",neutral


### Step 3 - Define Failure-Relevant Metadata

#TODO:
In this step, you will create **at least 5** metadata columns that help you slice and analyze model behavior in Weights & Biases (W&B).
These metadata columns should **capture meaningful properties of the data or model behavior that may influence performance**. You can define them using:

1. Value matching (e.g., tweets containing hashtags or mentions)
2. Regex patterns (e.g., negation words, strong sentiment terms like love or hate)
3. Heuristics (e.g., emoji count, all-caps text, tweet length buckets)

Each metadata column should correspond to a potential hypothesis about when or why a model might succeed or fail.
These columns will be propagated through inference and included in the final predictions_table logged to W&B.

After inference, your W&B table (df_long) will contain:
- The original tweet text
- Ground-truth sentiment labels
- Model predictions and confidence scores
- All metadata columns you defined for slicing

You will use these metadata fields in the W&B UI (via the ➕ Filter option) to:
- Create slices of the data
- Compare model behavior across slices
- Identify patterns, weaknesses, or regressions that are not visible in overall accuracy

In [17]:
# Replace or extend these examples with at least five hypothesis-driven slices.
def count_emojis(text: str) -> int:
    return sum(ch in emoji.EMOJI_DATA for ch in str(text))

df["emoji_count"] = df["text"].apply(count_emojis).astype(int)
df["has_hashtag"] = df["text"].str.contains(r"#\w+", regex=True)
df["has_mention"] = df["text"].str.contains(r"@\w+", regex=True)
df["has_negation"] = df["text"].str.contains(r"\b(?:not|never|no)\b", case=False, regex=True)
df["length_bucket"] = pd.cut(
    df["text"].str.len(),
    bins=[0, 50, 100, 200, 1000, 10_000],
    labels=["0-50", "51-100", "101-200", "201-1000", "1001+"],
    include_lowest=True
).astype(str)

df["happy"] = df["text"].str.contains(r"\b(?:happy|joy|smile|excited|eager)\b", case=False, regex=True)
df["sad"] = df["text"].str.contains(r"\b(?:sad|depressed|bad|dissapointed|upset)\b", case=False, regex=True)
df["trendy"] = df["text"].str.contains(r"\b(?:cool|popular|new|trend)\b", case=False, regex=True)
df["niche"] = df["text"].str.contains(r"\b(?:niche|underground|unknown|secret|hidden|unpopular)\b", case=False, regex=True)

def get_slices(df_any: pd.DataFrame):
    return {
        "has_emoji": df_any["emoji_count"] > 0,
        "has_negation": df_any["has_negation"] == True,
        "has_hashtag": df_any["has_hashtag"] == True,
        "has_mention": df_any["has_mention"] == True,
        "long_tweets": df_any["length_bucket"].isin(["101-200", "201-1000", "1001+"]),
        "sad": df_any["sad"] == True,
        "happy": df_any["happy"] == True,
        "trendy": df_any["trendy"] == True,
        "niche": df_any["niche"] == True,
        "short_tweets": df_any["length_bucket"].isin(["0-50"])
    }

In [18]:
assert len(get_slices(df)) >= 5, "Define at least five slices before continuing."
slice_sizes = {name: int(mask.sum()) for name, mask in get_slices(df).items()}
assert all(size > 0 for size in slice_sizes.values()), "Revise any empty slice before continuing."
pd.Series(slice_sizes, name="examples")

,examples
has_emoji,29
has_negation,43
has_hashtag,193
has_mention,208
long_tweets,218
sad,10
happy,7
trendy,18
niche,2
short_tweets,67


###  Step 4 – Run Inference (Two Models)

In this step, you'll use two HuggingFace sentiment analysis models to run inference on your dataset:

In [10]:
CLASSIFIERS = {}

def run_pipeline(model_id: str, texts: list[str]):
    if model_id not in CLASSIFIERS:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSequenceClassification.from_pretrained(model_id)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        if model.config.pad_token_id is None:
            model.config.pad_token_id = tokenizer.pad_token_id
        CLASSIFIERS[model_id] = pipeline(
            "text-classification",
            model=model,
            tokenizer=tokenizer,
            truncation=True,
            max_length=128,
            framework="pt",
            device=-1,
        )
    clf = CLASSIFIERS[model_id]
    outputs = clf(texts, batch_size=32)
    preds = [HF_LABEL_MAP.get(out["label"], out["label"]) for out in outputs]
    confs = [float(out["score"]) for out in outputs]
    return preds, confs

pred_frames = []
texts = df["text"].tolist()

for model_name, model_id in MODELS.items():
    yhat, conf = run_pipeline(model_id, texts)
    tmp = df.copy()
    tmp["model"] = model_name
    tmp["pred"] = yhat
    tmp["conf"] = conf
    pred_frames.append(tmp)

df_long = pd.concat(pred_frames, ignore_index=True)

# Add a stable example id so reshaping won't silently drop duplicates
df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

df_long.head(5)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

Device set to use cpu


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

,text,label,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,happy,sad,trendy,niche,model,pred,conf,ex_id
0,@user @user what do these '1/2 naked pics' hav...,neutral,0,False,True,True,51-100,False,False,False,False,baseline_model,negative,0.804726,113
1,OH: “I had a blue penis while I was this” [pla...,neutral,0,False,False,False,51-100,False,False,False,False,baseline_model,neutral,0.866949,363
2,"@user @user That's coming, but I think the vic...",neutral,0,False,True,False,51-100,False,False,False,False,baseline_model,neutral,0.763724,102
3,I think I may be finally in with the in crowd ...,positive,0,True,True,False,51-100,False,False,False,False,baseline_model,positive,0.774047,305
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative,0,False,True,False,101-200,False,False,False,False,baseline_model,neutral,0.416397,160


In [11]:
# Step 4.5 – Wide-format Table for Model Comparison (Optional but recommended)
# One row per tweet, with baseline + candidate predictions in columns
# TODO: Replace with your metadata
df_wide = df_long.pivot_table(
    index=[
        "ex_id", "text", "label",
        "emoji_count", "has_hashtag", "has_mention", "has_negation", "length_bucket"
    ],
    columns="model",
    values=["pred", "conf"],
    aggfunc="first"
).reset_index()

# Flatten column names (e.g., pred_baseline_model, conf_candidate_model)
df_wide.columns = ["_".join([c for c in col if c]).strip("_") for col in df_wide.columns]

df_wide.head(5)

,ex_id,text,label,emoji_count,has_hashtag,has_mention,has_negation,length_bucket,conf_baseline_model,conf_candidate_model,pred_baseline_model,pred_candidate_model
0,0,"""Fatty Kim The Third"" 😭😭😭",neutral,3,False,False,False,0-50,0.486252,0.978987,neutral,neutral
1,1,"""Focusing on [alt rightists'] respectability.....",neutral,0,False,False,False,51-100,0.573571,0.999728,negative,neutral
2,2,"""Kim Fatty the Third""",negative,0,False,False,False,0-50,0.849732,0.937708,neutral,neutral
3,3,"""We have lost everything"": Syrians return to r...",neutral,0,True,True,False,51-100,0.751955,0.994244,negative,positive
4,4,"""who's the most wiped out white boy? Zac Efron...",neutral,0,False,False,False,51-100,0.561233,0.906539,neutral,positive


### Step 5: Compute Metrics (Accuracy + Slice Accuracy + Regression)

In [12]:
# Compute overall and slice accuracy for the slices defined above.
from sklearn.metrics import accuracy_score

def compute_accuracy(y_true, y_pred):
    return accuracy_score(list(y_true), list(y_pred))

# Overall accuracy by model (df_long: one row per (tweet, model))
overall = df_long.groupby("model").apply(
    lambda g: compute_accuracy(g["label"], g["pred"]),
    include_groups=False
)

# Slice accuracy table (uses df_long masks)
slice_table = wandb.Table(columns=["slice", "model", "accuracy"])
slice_metrics = {}

for slice_name, mask in get_slices(df_long).items():
    slice_metrics[slice_name] = {}
    for model_name, g in df_long[mask].groupby("model"):
        if g.empty:
            continue
        acc = float(compute_accuracy(g["label"], g["pred"]))
        slice_table.add_data(slice_name, model_name, acc)
        slice_metrics[slice_name][model_name] = acc

pd.DataFrame(slice_metrics).T

,baseline_model,candidate_model
has_emoji,0.758621,0.517241
has_negation,0.651163,0.325581
has_hashtag,0.668394,0.461140
has_mention,0.697115,0.274038
long_tweets,0.660550,0.334862
sad,0.400000,0.100000
happy,1.000000,1.000000
trendy,0.833333,0.444444
niche,0.500000,0.000000
short_tweets,0.716418,0.552239


In [24]:
# TODO: Edit to work for your slices


# Regression-aware evaluation (df_eval: one row per tweet, both model outputs)
# A regression is when the candidate gets something wrong that the baseline got right.
BASELINE = "baseline_model"
CANDIDATE = "candidate_model"

# Ensure ex_id exists (safe even if it already exists)
df_long = df_long.copy()
if "ex_id" not in df_long.columns:
    df_long["ex_id"] = df_long.groupby(["text", "label"]).ngroup()

# Build df_eval with metadata carried through
df_eval = (
    df_long.pivot_table(
        index=[
            "ex_id", "text", "label",
            "emoji_count", "has_hashtag", "has_mention", "has_negation", "length_bucket",
            "sad", "happy", "trendy", "niche"
        ],
        columns="model",
        values=["pred", "conf"],
        aggfunc="first"
    )
    .reset_index()
)

# Flatten column names (pred_baseline_model, conf_candidate_model, etc.)
df_eval.columns = ["_".join([c for c in col if c]).strip("_") for col in df_eval.columns]

# Correctness flags
df_eval["baseline_correct"]  = df_eval[f"pred_{BASELINE}"] == df_eval["label"]
df_eval["candidate_correct"] = df_eval[f"pred_{CANDIDATE}"] == df_eval["label"]

# Regression / improvement flags
df_eval["regressed"]   = df_eval["baseline_correct"] & ~df_eval["candidate_correct"]
df_eval["improved"]    = ~df_eval["baseline_correct"] & df_eval["candidate_correct"]
df_eval["both_wrong"]  = ~df_eval["baseline_correct"] & ~df_eval["candidate_correct"]
df_eval["both_correct"]= df_eval["baseline_correct"] & df_eval["candidate_correct"]

# Confidence-conditional regression (candidate is confident AND worse than baseline)
df_eval["confident_regression"] = df_eval["regressed"] & (df_eval[f"conf_{CANDIDATE}"] >= 0.8)

# Global regression metrics
regression_rate = float(df_eval["regressed"].mean())
improvement_rate = float(df_eval["improved"].mean())
conf_reg_rate = float(df_eval["confident_regression"].mean())

print("Regression rate:", regression_rate)
print("Improvement rate:", improvement_rate)
print("Confident regression rate:", conf_reg_rate)

Regression rate: 0.408
Improvement rate: 0.108
Confident regression rate: 0.382


In [25]:
# Slice-level regression metrics table
reg_table = wandb.Table(columns=["slice", "metric", "value"])
reg_metrics = {}

for slice_name, mask in get_slices(df_eval).items():
    g = df_eval[mask]
    if len(g) == 0:
        continue

    reg = float(g["regressed"].mean())
    imp = float(g["improved"].mean())
    conf_reg = float(g["confident_regression"].mean())

    reg_table.add_data(slice_name, "regression_rate", reg)
    reg_table.add_data(slice_name, "improvement_rate", imp)
    reg_table.add_data(slice_name, "confident_regression_rate", conf_reg)

    reg_metrics[slice_name] = {
        "regression_rate": reg,
        "improvement_rate": imp,
        "conf_reg_rate": conf_reg
    }



### Step 6 — Log to W&B and analyse slices

Keep the run open until the final Step 7 cell so the optional synthetic table can be logged to the same run.

In [26]:
# Step 6: Log to W&B

run = wandb.init(project=PROJECT, entity=ENTITY, name=RUN_NAME)
run.log({"predictions_table": wandb.Table(dataframe=df_long)})
run.log({"slice_metrics": slice_table})
run.log({"regression_metrics": reg_table})
run.log({
    "df_eval": wandb.Table(dataframe=df_eval)
})
for model_name, acc in overall.items():
    run.summary[f"{model_name}_accuracy"] = float(acc)
run.summary["regression_rate"] = regression_rate
run.summary["improvement_rate"] = improvement_rate
run.summary["confident_regression_rate"] = conf_reg_rate

wandb_run_url = run.get_url()
print("W&B run URL:", wandb_run_url)

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


W&B run URL: https://wandb.ai/durb-carnegie-mellon-university/mlip-lab4-slices-2026/runs/nbczj435


### Instructions: Exploring Slice-Based Evaluation in W&B

# Purpose
In this lab, you are evaluating a candidate sentiment model to decide whether it should replace an existing baseline (production) model.
You have already:
  - run both models on the same dataset
  - logged predictions, confidence scores, and metadata to W&B
  - created metadata that allows you to slice the data
The most important goal is to understand when and why models behave differently.
Overall accuracy alone is often misleading.

# What to do in W&B
1. Open your W&B run
  - Click the project link and open the latest run.
2. Explore the predictions table
  - Go to the Tables tab and open predictions_table.
  - Each row is one tweet × one model.
3. Create and analyze slices (most important)
  - Use filters to create meaningful slices
    (e.g., negation, emojis, hashtags, long tweets).
  - For each slice:
    - Compare baseline vs candidate performance.
    - Compare slice accuracy to overall accuracy.
    - Inspect a few misclassified examples to identify patterns.
4. Visualize slice performance
  - Open slice_metrics.
  - Create bar charts comparing baseline vs candidate accuracy for at least two slices.
5. Record your findings in the notebook
  - Explain why slicing reveals issues that overall accuracy hides.
  - Say whether the candidate model should be deployed and why.


In [ ]:
# Students: replace the placeholders below with 1–2 sentence insights
#TODO: Replace this with 1-2 sentence takeaways for each slice.
saved_slice_notes = ["..."]
pd.DataFrame(saved_slice_notes)

### Step 7 - Targeted stress testing with LLMs

In this step, use an LLM to generate test cases that target a weakness you observed during slicing.

1. Choose one slice with poor performance, regressions, or surprising behaviour.
2. Write a one- or two-sentence hypothesis explaining why the model may struggle on this slice.
3. Generate ten subtle, difficult, adversarial, or minimally different tweets that test the hypothesis.
4. Assign each tweet an expected sentiment label before looking at either model's prediction.
5. Run both models on the ten labeled cases and inspect their accuracy and individual failures.
6. Record whether the same failures appeared, whether a new pattern emerged, and how the results affect your deployment judgment.

Set `USE_LOCAL_GENERATOR` to `True` to use the public `HuggingFaceTB/SmolLM2-360M-Instruct` model on the Colab CPU without an account, token, or paid API.
You may instead use a free LLM service available to you and paste only its generated text into the notebook.
Never paste an API token into a notebook cell.

In [31]:
# Replace the example hypothesis and cases with evidence targeting your chosen slice.
generated_slice_description = "The candidate fails to detect sad emotions as negative sentiments"
USE_LOCAL_GENERATOR = False

example_cases = [
# Negative (sadness-related)
{"text": "I miss my grandmother every day. Some losses never get easier.", "expected_label": "negative"},
{"text": "Feeling heartbroken after hearing the news this morning.", "expected_label": "negative"},
{"text": "The house feels so empty now that everyone is gone.", "expected_label": "negative"},
{"text": "I cried all night after saying goodbye at the airport.", "expected_label": "negative"},
{"text": "Today has been lonely and depressing, and I can't shake the feeling.", "expected_label": "negative"},

# Positive controls
{"text": "Just got promoted at work and I'm absolutely thrilled!", "expected_label": "positive"},
{"text": "Had an amazing weekend with friends and family.", "expected_label": "positive"},
{"text": "I finally finished my project and couldn't be happier with the result.", "expected_label": "positive"},
{"text": "Beautiful weather today. Feeling grateful and optimistic.", "expected_label": "positive"},
{"text": "Excited to start a new chapter in my life next week!", "expected_label": "positive"},
]

if USE_LOCAL_GENERATOR:
    local_generator = pipeline(
        "text-generation",
        model="HuggingFaceTB/SmolLM2-360M-Instruct",
        device=-1,
    )
    messages = [{
        "role": "user",
        "content": (
            "Write one short sentiment-analysis test tweet for this hypothesis: "
            f"{generated_slice_description} Return only the tweet."
        ),
    }]
    generated_texts = [
        item["generated_text"][-1]["content"].strip()
        for item in local_generator(
            messages,
            do_sample=True,
            max_new_tokens=48,
            num_return_sequences=10,
            temperature=0.9,
            top_p=0.95,
        )
    ]
    generated_cases = [{"text": text, "expected_label": ""} for text in generated_texts]
    print("Assign an expected label to every generated case before running the next cell.")
else:
    generated_cases = example_cases

pd.DataFrame(generated_cases)

,text,expected_label
0,I miss my grandmother every day. Some losses n...,negative
1,Feeling heartbroken after hearing the news thi...,negative
2,The house feels so empty now that everyone is ...,negative
3,I cried all night after saying goodbye at the ...,negative
4,"Today has been lonely and depressing, and I ca...",negative
5,Just got promoted at work and I'm absolutely t...,positive
6,Had an amazing weekend with friends and family.,positive
7,I finally finished my project and couldn't be ...,positive
8,Beautiful weather today. Feeling grateful and ...,positive
9,Excited to start a new chapter in my life next...,positive


In [32]:
def run_on_generated_tests(cases, models=MODELS):
    expected = pd.DataFrame(cases)
    assert len(expected) == 10, "Provide exactly ten generated cases."
    assert set(expected.columns) == {"text", "expected_label"}, "Each case needs text and expected_label."
    assert expected["text"].str.strip().ne("").all(), "Every case needs text."
    allowed_labels = {"negative", "neutral", "positive"}
    assert expected["expected_label"].isin(allowed_labels).all(), "Use negative, neutral, or positive labels."

    rows = []
    for model_name, model_id in models.items():
        predictions, confidences = run_pipeline(model_id, expected["text"].tolist())
        for case, prediction, confidence in zip(cases, predictions, confidences):
            rows.append({
                "text": case["text"],
                "expected_label": case["expected_label"],
                "model": model_name,
                "pred": prediction,
                "conf": confidence,
                "correct": prediction == case["expected_label"],
            })
    return pd.DataFrame(rows)


In [33]:
generated_df = run_on_generated_tests(generated_cases)
stress_metrics = generated_df.groupby("model")["correct"].mean().rename("accuracy")
display(stress_metrics)
generated_df

,accuracy
model,
baseline_model,1.0
candidate_model,0.3


,text,expected_label,model,pred,conf,correct
0,I miss my grandmother every day. Some losses n...,negative,baseline_model,negative,0.862145,True
1,Feeling heartbroken after hearing the news thi...,negative,baseline_model,negative,0.883324,True
2,The house feels so empty now that everyone is ...,negative,baseline_model,negative,0.823543,True
3,I cried all night after saying goodbye at the ...,negative,baseline_model,negative,0.776746,True
4,"Today has been lonely and depressing, and I ca...",negative,baseline_model,negative,0.931956,True
5,Just got promoted at work and I'm absolutely t...,positive,baseline_model,positive,0.990482,True
6,Had an amazing weekend with friends and family.,positive,baseline_model,positive,0.986911,True
7,I finally finished my project and couldn't be ...,positive,baseline_model,positive,0.988398,True
8,Beautiful weather today. Feeling grateful and ...,positive,baseline_model,positive,0.986356,True
9,Excited to start a new chapter in my life next...,positive,baseline_model,positive,0.988593,True


In [34]:
# Log the synthetic evidence to the active run, then close the run cleanly.
run.log({
    "synthetic_tests": wandb.Table(dataframe=generated_df)
})
for model_name, accuracy in stress_metrics.items():
    run.summary[f"synthetic_{model_name}_accuracy"] = float(accuracy)
print("W&B run URL:", wandb_run_url)
run.finish()

W&B run URL: https://wandb.ai/durb-carnegie-mellon-university/mlip-lab4-slices-2026/runs/nbczj435


baseline_model_accuracy,0.698
candidate_model_accuracy,0.398
confident_regression_rate,0.382
improvement_rate,0.108
regression_rate,0.408
synthetic_baseline_model_accuracy,1
synthetic_candidate_model_accuracy,0.3
